# *LLM as a Service* (Gemini AI) - Peringkasan Berita

---

Digunakan pendekatan LLMaaS dengan menggunakan model Gemini 3.5 Flash-Lite, sebagai perbandingan performa *pre-trained model*.

In [ ]:
import time
import numpy as np
import pandas as pd
from google import genai
from google.genai import types
from google.genai.errors import APIError

In [ ]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# fallback api key
if not GEMINI_API_KEY:
    import getpass
    GEMINI_API_KEY = getpass.getpass('Masukkan Gemini API Key: ')

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_NAME = 'gemini-3.5-flash-lite'

CALL_DELAY = 4  # 4 detik jeda panggilan

print('Client Gemini model:', MODEL_NAME)

Client Gemini model: gemini-3.5-flash-lite


In [ ]:
import gdown
import pandas as pd

url_rawdata = "https://docs.google.com/spreadsheets/d/1pstC3_51qAytcffcvl-lfGnHr_RbOPK3/export?format=xlsx"
gdown.download(url_rawdata, "humaneval_data.xlsx", quiet=False, fuzzy=True)

df_he = pd.read_excel('humaneval_data.xlsx')
print('Shape:', df_he.shape)
print('Kolom:', df_he.columns.tolist())
print('\nNull values:')
print(df_he.isnull().sum())
print('\nContoh data:')
df_he.head(3)

HUMANEVAL_PATH = 'humaneval_data.xlsx'
TEXT_COLUMN = 'Ringkasan'

df_he = pd.read_excel(HUMANEVAL_PATH)
print('Shape:', df_he.shape)
print('Kolom:', df_he.columns.tolist())
df_he.head(3)

Downloading...
From: https://docs.google.com/spreadsheets/d/1pstC3_51qAytcffcvl-lfGnHr_RbOPK3/export?format=xlsx
To: /content/humaneval_data.xlsx
51.2kB [00:00, 51.0MB/s]


Shape: (30, 6)
Kolom: ['Judul', 'Link', 'Ringkasan', 'Full_Content', 'Label', 'Waktu']

Null values:
Judul           0
Link            0
Ringkasan       0
Full_Content    0
Label           0
Waktu           0
dtype: int64

Contoh data:
Shape: (30, 6)
Kolom: ['Judul', 'Link', 'Ringkasan', 'Full_Content', 'Label', 'Waktu']


,Judul,Link,Ringkasan,Full_Content,Label,Waktu
0,Federasi Ekspor Pangan India Gelar ‘Buyer and ...,https://mediaindonesia.com/ekonomi/645883/fede...,"Prashant Seth, Juru Bicara Deputi FIEO (Federa...",Pertemuan ‘Buyer and Seller Meet’ yang digelar...,EI,01-2024
1,"Warga RI Makan Mie Miliaran Bungkus, Impor Gan...",https://www.cnbcindonesia.com/research/2024012...,Data Badan Pusat Statistik (BPS) menunjukkan n...,"Jakarta, CNBC Indonesia - Impor gandum dan mes...",EI,01-2024
2,"Impor Pecah Rekor - Produksi Ambruk, Apa Kabar...",https://www.cnbcindonesia.com/research/2024011...,"Berdasarkan data BPS, angka sementara produksi...","Jakarta, CNBC Indonesia - Swasembada pangan te...",EI,01-2024


In [ ]:
# call gemini
def _call_gemini(prompt, max_retries=5):
    """Panggil Gemini dengan retry + exponential backoff untuk error sementara
    (503 server sibuk, rate limit, koneksi putus, dll)."""
    last_error = None
    for attempt in range(max_retries):
        try:
            interaction = client.interactions.create(
                model=MODEL_NAME,
                input=prompt,
            )
            return interaction.output_text.strip()
        except Exception as e:
            last_error = e
            wait = min(10 * (2 ** attempt), 120)  # exponential backoff, dibatasi maks 120s
            print(f'  Error ({type(e).__name__}: {e}), coba lagi dalam {wait}s... (percobaan {attempt+1}/{max_retries})')
            time.sleep(wait)
    raise RuntimeError(f'Gagal setelah {max_retries} percobaan.') from last_error

# prompt SDS + berita
def generate_summary_sds(text):
    prompt = (
        'Anda adalah asisten yang meringkas berita berbahasa Indonesia. '
        'Ringkas artikel berikut dalam satu paragraf singkat (sepanjang 3-4 kalimat), '
        'gaya bahasa berita formal. Pastikan memiliki kefasihan, kandungan informasi, dan non-redundansi terjaga. '
        'Jangan tambahkan kalimat pembuka, penutup, atau komentar apapun, cukup hasil ringkasannya saja.\n\n'
        f'Artikel:\n{text}'
    )
    return _call_gemini(prompt)

# prompt MDS + set ringkasan
def generate_summary_mds(list_ringkasan):
    ringkasan_bernomor = '\n\n'.join(
        f'Berita {i+1}: {r}' for i, r in enumerate(list_ringkasan)
    )
    prompt = (
        'Anda adalah asisten yang menggabungkan beberapa ringkasan berita berbahasa Indonesia '
        'yang membahas indikator/topik yang sama menjadi SATU ringkasan akhir yang utuh. '
        'Ringkas poin-poin penting dari seluruh ringkasan di bawah ini, jangan hanya '
        'mengambil atau menyalin salah satu ringkasan saja. Tulis dalam satu paragraf dengan '
        'gaya bahasa berita formal. Pastikan memiliki kefasihan, kandungan informasi, dan non-redundansi terjaga. '
        'Jangan tambahkan kalimat pembuka, penutup, atau komentar apapun cukup hasil ringkasan akhirnya saja.\n\n'
        f'{ringkasan_bernomor}'
    )
    return _call_gemini(prompt)


def timed_call(fn, *args):
    start = time.perf_counter()
    hasil = fn(*args)
    duration = time.perf_counter() - start
    return hasil, duration

In [ ]:
# Generate SDS
texts_he = df_he[TEXT_COLUMN].tolist()

ringkasan_sds = []
durasi_sds = []

print(f'Generate SDS untuk {len(texts_he)} berita...')
for i, text in enumerate(texts_he):
    hasil, durasi = timed_call(generate_summary_sds, text)
    ringkasan_sds.append(hasil)
    durasi_sds.append(durasi)
    print(f'[{i+1}/{len(texts_he)}] {durasi:.2f}s -> {hasil[:70]}...')
    time.sleep(CALL_DELAY)

durasi_sds = np.array(durasi_sds)
print(f'\nRata-rata runtime SDS: {durasi_sds.mean():.2f}s | Total: {durasi_sds.sum():.2f}s')

df_he['Ringkasan_Gemini_SDS'] = ringkasan_sds
df_he['Runtime_Gemini_SDS_s'] = durasi_sds

Generate SDS untuk 30 berita...
[1/30] 1.64s -> Federation of Indian Export Organisations (FIEO) mencatat Indonesia se...
[2/30] 1.42s -> Badan Pusat Statistik mencatat volume impor gandum dan meslin Indonesi...
[3/30] 1.48s -> Badan Pusat Statistik (BPS) mencatat angka sementara produksi beras na...
[4/30] 1.66s -> Sektor industri manufaktur Indonesia pada bulan Mei masih berada dalam...
[5/30] 1.70s -> Pemberlakuan larangan ekspor minyak sawit mentah (CPO) oleh pemerintah...
[6/30] 1.66s -> Realisasi pendapatan Anggaran Pendapatan dan Belanja Negara (APBN) Reg...
[7/30] 1.45s -> Kenaikan harga tiket pesawat saat ini mendorong peningkatan tren libur...
[8/30] 1.81s -> Menteri Pariwisata dan Ekonomi Kreatif Sandiaga Salahuddin Uno mencata...
[9/30] 1.67s -> Indonesia Investment Forum di New York akan berfokus membahas pemetaan...
[10/30] 1.32s -> Menteri Pariwisata dan Ekonomi Kreatif Sandiaga Uno memastikan bahwa k...
[11/30] 1.57s -> Maskapai Garuda Indonesia menyelenggarakan program

In [ ]:
# MDS per set berita
num_articles = len(df_he)

num_sets = num_articles // 3
ringkasan_mds = []
durasi_mds = []
copy_verbatim_flags = []

print(f'Generate MDS untuk {num_sets} set...')
for i in range(0, num_articles, 3):
    set_sds = df_he['Ringkasan_Gemini_SDS'].iloc[i:i+3].tolist()
    if len(set_sds) < 3:
        continue

    hasil, durasi = timed_call(generate_summary_mds, set_sds)
    ringkasan_mds.append(hasil)
    durasi_mds.append(durasi)

    is_copy = any(hasil.strip() == s.strip() for s in set_sds)
    copy_verbatim_flags.append(is_copy)
    flag_text = '  PERINGATAN: COPY VERBATIM' if is_copy else ''
    print(f'Set {i//3 + 1}/{num_sets}: {durasi:.2f}s -> {hasil[:70]}...{flag_text}')

    time.sleep(CALL_DELAY)

durasi_mds = np.array(durasi_mds)
persentase_copy = 100 * sum(copy_verbatim_flags) / len(copy_verbatim_flags)
print(f'\nRata-rata runtime MDS: {durasi_mds.mean():.2f}s | Total: {durasi_mds.sum():.2f}s')
print(f'Persentase set copy verbatim: {persentase_copy:.1f}%')

df_he['Ringkasan_Gemini_MDS'] = ''
df_he['Runtime_Gemini_MDS_s'] = np.nan

for j in range(len(ringkasan_mds)):
    row_idx = j * 3
    if row_idx < len(df_he):
        df_he.at[row_idx, 'Ringkasan_Gemini_MDS'] = ringkasan_mds[j]
        df_he.at[row_idx, 'Runtime_Gemini_MDS_s'] = durasi_mds[j]

Generate MDS untuk 10 set...
Set 1/10: 2.24s -> Dinamika sektor pangan dan perdagangan nasional menunjukkan peningkata...
Set 2/10: 2.03s -> Sektor industri manufaktur Indonesia pada Mei 2022 tetap berada di zon...
Set 3/10: 1.62s -> Kenaikan harga tiket pesawat telah mendorong peningkatan tren wisata j...
Set 4/10: 1.68s -> Menteri Pariwisata dan Ekonomi Kreatif Sandiaga Uno memastikan kebijak...
Set 5/10: 2.34s -> Kementerian PPN/Bappenas memproyeksikan pertumbuhan ekonomi Indonesia ...
Set 6/10: 1.81s -> APBN 2024 didesain sebagai instrumen fiskal yang antisipatif, responsi...
Set 7/10: 1.60s -> Badan Pusat Statistik mencatat inflasi Jakarta pada Desember 2023 menc...
Set 8/10: 1.60s -> Dinas KPKP DKI Jakarta optimistis inflasi daerah tetap terkendali di b...
Set 9/10: 1.37s -> Dinas Kesehatan DKI Jakarta memprediksi kasus Covid-19 sulit mencapai ...
Set 10/10: 1.53s -> Kementerian Ketenagakerjaan mencatat 7.469 pekerja di Jakarta, yang di...

Rata-rata runtime MDS: 1.78s | Total: 1

In [ ]:
# simpan hasil sds dan mds ke excel
df_he.to_excel('humaneval_gemini.xlsx', index=False)